In [52]:
%load_ext autoreload
%autoreload 2

import sys
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append("/root/limlab01/kaistai/25DFT/QHFlow/src")
import md.scflow_calculator_gpu
from dft_process.dft_process_utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
# data_index = 1 is ethanol
md17_experiment = MD17Experiment(data_index=3, use_shard=True)

md17_experiment.set_model_path()

INFO     [setup.py:66] >> Seed: 0

Seed set to 0


ConfigNamespace({'dataset_name': 'water', 'num_train': 500, 'num_valid': 500, 'density_loss': 0.01, 'max_radius': 15.0, 'batch_size': 16, 'train_batch_size': 16, 'valid_batch_size': 16, 'test_batch_size': 16})


INFO     [dft_process_utils.py:204] >> [!] conf.dataset.dataset_name: uracil

md17
uracil
MD17_DFT(30000)


In [54]:
md17_experiment.dataset[0]

AOData(pos=[12, 3], atoms=[12, 1], energy=[1], force=[12, 3], AO_index=[3, 132], hamiltonian=[1, 132, 132], init_ham=[1, 132, 132], overlap=[1, 132, 132], num_atoms=[1], AO_l_index=[60], AO_l_index_len=[1, 1], full_edge_index=[2, 132], mask_row=[12, 14], Q=[132, 132, 60])

In [55]:
print(md17_experiment.dataset[0].energy * (HA2eV))
print(md17_experiment.dataset[0].force * (HA2eV / BOHR2ANG))
print(md17_experiment.dataset[0].pos * BOHR2ANG)

tensor([-11266.7043])
tensor([[-3.5659e+00, -1.2219e+00,  2.5053e+00],
        [-7.9203e-01,  2.0875e+00,  3.4962e-01],
        [ 5.8883e-01,  2.1348e+00, -5.5762e-01],
        [ 5.4165e+00, -1.0356e-01,  4.7755e-02],
        [ 3.0840e+00, -2.2277e+00, -1.4367e+00],
        [ 5.1807e+00,  4.5001e+00,  5.6024e-01],
        [-4.0101e+00, -1.2652e+00, -1.5983e-01],
        [-2.5472e+00, -5.5717e+00,  2.5886e-02],
        [ 5.4775e-01, -4.8231e-01, -8.3750e-01],
        [-8.4020e-01,  2.0211e-01,  4.5042e-01],
        [-2.0556e+00,  1.9519e+00,  1.6449e-02],
        [-1.0068e+00, -3.8643e-03, -9.6399e-01]])
tensor([[ 1.6804,  0.2498, -0.1868],
        [ 1.4016, -1.0784,  0.1033],
        [ 0.0971, -1.5801,  0.1753],
        [-1.0739, -0.7439,  0.0974],
        [-0.7741,  0.6325,  0.1935],
        [ 0.4782,  1.2083, -0.1512],
        [-2.1678, -1.1964, -0.0237],
        [ 0.6880,  2.4570, -0.2299],
        [ 2.1627, -1.7686,  0.4871],
        [-0.0096, -2.5829, -0.0028],
        [-1.5787,  

In [56]:
from dataset_module.ori_dataset import random_split

In [57]:
dummy_data = torch.arange(len(md17_experiment.dataset))
print(len(dummy_data))
splits = random_split(dummy_data, [25000,500,len(dummy_data)-25500],42)

30000


In [58]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def convert_dataset_to_csv(dataset, splits, molecule_name, output_dir="./"):
    """
    Convert MD17 dataset to CSV format (single combined file).
    
    Args:
        dataset: MD17 dataset object
        splits: List of three Subset objects [train, val, test]
        molecule_name: Name of the molecule (e.g., 'ethanol')
        output_dir: Output directory for CSV files
    """
    os.makedirs(output_dir, exist_ok=True)
    
    split_names = ['train', 'val', 'test']
    all_rows = []
    sample_data = None  # Store first frame for sample file
    
    print(f"\nProcessing all splits and combining into single CSV...")
    
    for split_name, split_dataset in zip(split_names, splits):
        print(f"\nProcessing {split_name} split ({len(split_dataset)} frames)...")
        
        for idx in tqdm(range(len(split_dataset)), desc=f"Converting {split_name}"):
            # Get the actual index in the full dataset
            # Subset objects have an 'indices' attribute that maps to the original dataset
            if hasattr(split_dataset, 'indices'):
                frame_idx = split_dataset.indices[idx]
            else:
                # Fallback: use the index directly
                frame_idx = idx
            
            # Use Subset's __getitem__ which handles indexing correctly
            data = split_dataset[idx]
            
            # Get data attributes
            pos = data.pos.cpu().numpy() if hasattr(data.pos, 'cpu') else np.array(data.pos)
            force = data.force.cpu().numpy() if hasattr(data.force, 'cpu') else np.array(data.force)
            
            # Get energy - try different possible attribute names
            energy = data.energy.cpu().numpy() if hasattr(data.energy, 'cpu') else np.array(data.energy)

            # Get atomic numbers
            if hasattr(data, 'z'):
                atomic_numbers = data.z.cpu().numpy() if hasattr(data.z, 'cpu') else np.array(data.z)
            elif hasattr(data, 'atoms'):
                atomic_numbers = data.atoms.cpu().numpy() if hasattr(data.atoms, 'cpu') else np.array(data.atoms)
            elif hasattr(data, 'atomic_numbers'):
                atomic_numbers = data.atomic_numbers.cpu().numpy() if hasattr(data.atomic_numbers, 'cpu') else np.array(data.atomic_numbers)
            else:
                raise AttributeError(f"Data object does not have 'z', 'atoms', or 'atomic_numbers' attribute. Available attributes: {[attr for attr in dir(data) if not attr.startswith('_')]}")
            
            # Convert units
            # Positions: Bohr to Angstrom
            pos_ang = pos * BOHR2ANG
            
            # Forces: Hartree/Bohr to eV/Angstrom
            force_ev_ang = force * (HA2eV / BOHR2ANG)
            
            # Energy: Hartree to eV
            energy_ev = energy * HA2eV
            
            # Get total energy (should be a scalar)
            if energy_ev.ndim > 0:
                energy_ev = energy_ev[0] if len(energy_ev) > 0 else energy_ev.item()
            else:
                energy_ev = float(energy_ev)
            
            # Store sample data from first frame
            if sample_data is None:
                sample_data = {
                    'frame_id': frame_idx,
                    'split': split_name,
                    'num_atoms': len(atomic_numbers),
                    'atomic_numbers': atomic_numbers.tolist(),
                    'positions_ang': pos_ang.tolist(),
                    'forces_ev_ang': force_ev_ang.tolist(),
                    'energy_ev': energy_ev
                }
            
            # Create rows for each atom
            num_atoms = len(atomic_numbers)
            for atom_id in range(num_atoms):
                row = {
                    'frame_id': frame_idx,
                    'atom_id': atom_id,
                    'atomic_number': int(atomic_numbers[atom_id]),
                    'x': float(pos_ang[atom_id, 0]),
                    'y': float(pos_ang[atom_id, 1]),
                    'z': float(pos_ang[atom_id, 2]),
                    'fx': float(force_ev_ang[atom_id, 0]),
                    'fy': float(force_ev_ang[atom_id, 1]),
                    'fz': float(force_ev_ang[atom_id, 2]),
                    'E_total': energy_ev
                }
                all_rows.append(row)
    
    # Create DataFrame from all rows
    print(f"\nCreating combined DataFrame...")
    df = pd.DataFrame(all_rows)
    
    # Sort by frame_id and atom_id
    df = df.sort_values(['frame_id', 'atom_id']).reset_index(drop=True)
    
    # Save to single CSV file
    output_filename = f"md17-mlff-{molecule_name}.csv"
    output_path = os.path.join(output_dir, output_filename)
    df.to_csv(output_path, index=False)
    print(f"Saved {output_filename}: {len(df)} rows ({len(df) // sample_data['num_atoms']} frames)")
    
    # Save sample file as txt
    sample_filename = f"md17-mlff-{molecule_name}-sample.txt"
    sample_path = os.path.join(output_dir, sample_filename)
    with open(sample_path, 'w') as f:
        f.write(f"Sample Data from MD17 Dataset: {molecule_name}\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Frame ID: {sample_data['frame_id']}\n")
        f.write(f"Split: {sample_data['split']}\n")
        f.write(f"Number of atoms: {sample_data['num_atoms']}\n")
        f.write(f"Total Energy (eV): {sample_data['energy_ev']:.6f}\n\n")
        f.write("Atomic Numbers:\n")
        f.write(f"  {sample_data['atomic_numbers']}\n\n")
        f.write("Positions (Angstrom):\n")
        for i, pos in enumerate(sample_data['positions_ang']):
            f.write(f"  Atom {i} (Z={sample_data['atomic_numbers'][i]}): [{pos[0]:10.6f}, {pos[1]:10.6f}, {pos[2]:10.6f}]\n")
        f.write("\nForces (eV/Angstrom):\n")
        for i, force in enumerate(sample_data['forces_ev_ang']):
            f.write(f"  Atom {i} (Z={sample_data['atomic_numbers'][i]}): [{force[0]:10.6f}, {force[1]:10.6f}, {force[2]:10.6f}]\n")
    print(f"Saved sample file: {sample_filename}")
    
    print(f"\nAll files saved to {output_dir}")



In [59]:
# Split dataset into train/val/test (25000/500/4500)
print(f"Total dataset size: {len(md17_experiment.dataset)}")
splits = random_split(md17_experiment.dataset, [25000, 500, len(md17_experiment.dataset)-25500], seed=42)

print(f"Train size: {len(splits[0])}")
print(f"Val size: {len(splits[1])}")
print(f"Test size: {len(splits[2])}")

# Get molecule name
molecule_name = md17_experiment.dataset_name

# Create output directory with molecule name
output_dir = f"./md17_processed/{molecule_name}"
os.makedirs(output_dir, exist_ok=True)

# Save split indices
split_indices = {
    'train': splits[0].indices.tolist() if hasattr(splits[0], 'indices') else list(range(len(splits[0]))),
    'val': splits[1].indices.tolist() if hasattr(splits[1], 'indices') else list(range(len(splits[0]), len(splits[0]) + len(splits[1]))),
    'test': splits[2].indices.tolist() if hasattr(splits[2], 'indices') else list(range(len(splits[0]) + len(splits[1]), len(splits[0]) + len(splits[1]) + len(splits[2])))
}

import json
split_indices_path = os.path.join(output_dir, f"md17-mlff-{molecule_name}-split-indices.json")
with open(split_indices_path, 'w') as f:
    json.dump(split_indices, f, indent=2)
print(f"\nSplit indices saved to {split_indices_path}")

# Convert to CSV (single combined file)
convert_dataset_to_csv(md17_experiment.dataset, splits, molecule_name, output_dir=output_dir)



Total dataset size: 30000
Train size: 25000
Val size: 500
Test size: 4500

Split indices saved to ./md17_processed/uracil/md17-mlff-uracil-split-indices.json

Processing all splits and combining into single CSV...

Processing train split (25000 frames)...


Converting train:   0%|          | 41/25000 [00:00<01:01, 402.88it/s]

Converting train: 100%|██████████| 25000/25000 [00:51<00:00, 489.67it/s]



Processing val split (500 frames)...


Converting val: 100%|██████████| 500/500 [00:00<00:00, 575.61it/s]



Processing test split (4500 frames)...


Converting test: 100%|██████████| 4500/4500 [00:09<00:00, 487.70it/s]



Creating combined DataFrame...
Saved md17-mlff-uracil.csv: 360000 rows (30000 frames)
Saved sample file: md17-mlff-uracil-sample.txt

All files saved to ./md17_processed/uracil


In [60]:
HA2eV

27.211396641308